In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import rcParams
import warnings
import traceback
import os  # 用于文件和目录操作

# 机器学习与预处理库
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb

# 忽略警告
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------
# 0. 配置输出目录
# ------------------------------------------------------------------
output_dir = '机器'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已创建输出目录: {output_dir}")
else:
    print(f"输出目录已存在: {output_dir}")

# ------------------------------------------------------------------
# 1. 字体与绘图环境全局设置
# ------------------------------------------------------------------
config = {
    "font.family": 'serif',
    "font.serif": ['Times New Roman'],
    "mathtext.fontset": 'stix',
    "axes.unicode_minus": False
}
rcParams.update(config)

# --- 字体定义 ---
chinese_font_name = 'SimSun'
english_font_name = 'Times New Roman'
label_fontsize, tick_fontsize, tick_fontsize_large, legend_fontsize, val_fontsize = 16, 14, 18, 14, 12

try:
    songti_font_label = fm.FontProperties(family=chinese_font_name, size=label_fontsize)
    songti_font_legend = fm.FontProperties(family=chinese_font_name, size=legend_fontsize)
    times_font_label = fm.FontProperties(family=english_font_name, size=label_fontsize)
    times_font_tick = fm.FontProperties(family=english_font_name, size=tick_fontsize)
    times_font_tick_large = fm.FontProperties(family=english_font_name, size=tick_fontsize_large)
    times_font_val = fm.FontProperties(family=english_font_name, size=val_fontsize, weight='bold')
    print(f"字体加载成功: {chinese_font_name} & {english_font_name}")
except:
    print("未检测到指定字体，使用默认字体。")
    songti_font_label, songti_font_legend = fm.FontProperties(size=label_fontsize), fm.FontProperties(size=legend_fontsize)
    times_font_label, times_font_tick, times_font_tick_large, times_font_val = fm.FontProperties(size=label_fontsize), fm.FontProperties(size=tick_fontsize), fm.FontProperties(size=tick_fontsize_large), fm.FontProperties(size=val_fontsize, weight='bold')

# ------------------------------------------------------------------
# 2. 样式辅助函数
# ------------------------------------------------------------------
def apply_publication_style(ax, large_ticks=False):
    bwith = 1.5
    for spine in ax.spines.values():
        spine.set_linewidth(bwith)
    ax.minorticks_on()
    ax.tick_params(axis="both", which="major", direction="in", width=1, length=5, pad=5)
    ax.tick_params(axis="both", which="minor", direction="in", width=1, length=2.5)
    current_tick_font = times_font_tick_large if large_ticks else times_font_tick
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontproperties(current_tick_font)
    ax.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha=0.6)
    ax.grid(True, which="minor", linestyle=":", color="lightgray", linewidth=0.75, alpha=0.5)

def set_mixed_labels(ax, x_zh, x_en, y_zh, y_en):
    ax.set_xlabel(''), ax.set_ylabel('')
    if x_zh == "":
        ax.text(0.5, -0.13, x_en, ha='center', va='center', transform=ax.transAxes, fontproperties=times_font_label)
    else:
        ax.text(0.5, -0.13, x_zh, ha='right', va='center', transform=ax.transAxes, fontproperties=songti_font_label)
        ax.text(0.5, -0.13, x_en, ha='left', va='center', transform=ax.transAxes, fontproperties=times_font_label)
    label_x_pos = -0.07
    if y_zh == "":
        ax.text(label_x_pos, 0.5, y_en, ha='center', va='center', rotation='vertical', transform=ax.transAxes, fontproperties=times_font_label)
    else:
        ax.text(label_x_pos, 0.5, y_en, ha='center', va='bottom', rotation='vertical', transform=ax.transAxes, fontproperties=times_font_label)
        ax.text(label_x_pos, 0.5, y_zh, ha='center', va='top', rotation='vertical', transform=ax.transAxes, fontproperties=songti_font_label)

# ------------------------------------------------------------------
# 3. 绘图函数定义 (修改为直接接收完整文件名)
# ------------------------------------------------------------------
def draw_metrics_radar(save_filename, metrics, color_theme):
    labels = ['$R^2$', 'MAE', 'MSE', 'RMSE', 'MAPE (%)', 'Cv(RMSE) (%)']
    raw_values = [metrics['R2'], metrics['MAE'], metrics['MSE'], metrics['RMSE'], metrics['MAPE'], metrics['Cv(RMSE)']]
    plot_values = [
        metrics['R2'], 1 / (1 + metrics['MAE'] / 10), 1 / (1 + metrics['MSE'] / 100),
        1 / (1 + metrics['RMSE'] / 10), 1 / (1 + metrics['MAPE'] / 20),
        1 / (1 + metrics['Cv(RMSE)'] / 30)
    ]
    N = 6
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
    plot_values_closed = np.concatenate((plot_values, [plot_values[0]]))
    angles_closed = np.concatenate((angles, [angles[0]]))
    
    fig = plt.figure(figsize=(6, 6), dpi=300)
    ax = fig.add_subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
    ax.grid(False); ax.spines['polar'].set_visible(False)
    
    for g in [0.2, 0.4, 0.6, 0.8, 1.0]:
        ax.plot(angles_closed, [g] * (N + 1), '-', lw=0.6, color='gray', alpha=0.5, linestyle='--')
    for ang in angles:
        ax.plot([ang, ang], [0, 1], ':', lw=0.6, color='gray', alpha=0.5)
    
    ax.plot(angles_closed, plot_values_closed, linewidth=2, color=color_theme, marker='o', markersize=5)
    ax.fill(angles_closed, plot_values_closed, color=color_theme, alpha=0.25)
    
    for i in range(N):
        angle_rad, val_plot, val_raw = angles[i], plot_values[i], raw_values[i]
        txt_str = f"{val_raw:.2f}%" if '%' in labels[i] else (f"{val_raw:.4f}" if labels[i] == '$R^2$' else f"{val_raw:.3f}")
        angle_deg = np.degrees(angle_rad) % 360
        ha = 'center' if angle_deg in [0, 180] else ('left' if 0 < angle_deg < 180 else 'right')
        va = 'top' if angle_deg == 0 else ('bottom' if angle_deg == 180 else 'center')
        offset = -0.1 if val_plot > 0.3 else 0.1
        ax.text(angle_rad, val_plot + offset, txt_str, fontproperties=times_font_val, color=color_theme, ha=ha, va=va)
        
    ax.set_xticks(angles); ax.set_xticklabels(labels, fontproperties=times_font_tick)
    ax.tick_params(pad=10); ax.set_yticks([]); ax.set_ylim(0, 1.05)
    plt.tight_layout()
    # 保存到文件夹，使用传入的具体文件名
    plt.savefig(os.path.join(output_dir, save_filename), dpi=300)
    plt.close()

def draw_regression_scatter(save_filename, y_true, y_pred, color_theme):
    fig, ax = plt.subplots(figsize=(7, 6), dpi=300)
    apply_publication_style(ax)
    ax.scatter(y_true, y_pred, alpha=0.6, s=35, facecolors='none', edgecolors=color_theme, lw=1)
    d_min, d_max = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.plot([d_min, d_max], [d_min, d_max], 'k--', lw=1.5)
    set_mixed_labels(ax, "实际", " HCLM (W/m)", "预测", " HCLM (W/m)")
    r2 = r2_score(y_true, y_pred)
    ax.text(0.05, 0.9, f'$R^2 = {r2:.4f}$', transform=ax.transAxes, fontproperties=fm.FontProperties(family=english_font_name, size=18))
    plt.tight_layout()
    # 保存到文件夹，使用传入的具体文件名
    plt.savefig(os.path.join(output_dir, save_filename), dpi=300)
    plt.close()

def draw_case_comparison(save_filename, y_true, y_pred, color_theme, y_range=None):
    subset = 30
    yt = y_true.values[:subset] if hasattr(y_true, 'values') else y_true[:subset]
    yp = y_pred[:subset]
    x = np.arange(1, subset + 1)
    
    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
    apply_publication_style(ax, large_ticks=True); ax.minorticks_off()
    
    ax.plot(x, yt, linestyle='--', color='gray', linewidth=1.2, label='实际值')
    ax.scatter(x, yt, marker='^', s=60, facecolors='none', edgecolors='gray', zorder=5)
    
    ax.plot(x, yp, linestyle='-', color=color_theme, linewidth=1.5, label='预测值')
    ax.scatter(x, yp, marker='s', s=60, facecolors='none', edgecolors=color_theme, zorder=5)
    
    set_mixed_labels(ax, "样本编号", "", "", "HCLM (W/m)")
    
    if y_range:
        ax.set_ylim(y_range)
    
    ax.set_xticks(np.arange(1, subset + 1, 2))
    ax.legend(prop=songti_font_legend, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2)
    plt.tight_layout()
    # 保存到文件夹，使用传入的具体文件名
    plt.savefig(os.path.join(output_dir, save_filename), dpi=300)
    plt.close()

# ------------------------------------------------------------------
# 4. 数据处理与主程序
# ------------------------------------------------------------------
try:
    # 1. 读取数据
    data_path = 'JQXX.csv'
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"未找到数据文件: {data_path}")
        
    data = pd.read_csv(data_path)
    
    print("-" * 50)
    print("【1. 原始数据统计】")
    print(f"原始数据形状 (行, 列): {data.shape}")
    print(f"原始缺失值总数: {data.isnull().sum().sum()}")
    print("-" * 50)

    # 2. 数据清洗
    original_rows = len(data)
    
    # 2.1 物理范围过滤
    data = data[(data['HCLM'] >= 0) & (data['HCLM'] <= 120)]
    rows_after_filter = len(data)
    
    # 2.2 替换无穷大值
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # 2.3 删除目标变量缺失的行
    data.dropna(subset=['HCLM'], inplace=True)
    rows_after_dropna = len(data)
    
    print("【2. 数据清洗报告】")
    print(f"过滤物理范围不合理行数: {original_rows - rows_after_filter}")
    print(f"删除目标值(HCLM)缺失行数: {rows_after_filter - rows_after_dropna}")
    print(f"清洗后数据形状 (行, 列): {data.shape}")
    print(f"剩余有效样本数: {len(data)}")
    print("-" * 50)
    
    if data.empty:
        raise ValueError("经过清洗后数据为空，请检查原始数据质量。")

    # 3. 定义特征与目标
    drop_cols = ['Run_ID', 'HCLM']
    cols_to_drop = [c for c in drop_cols if c in data.columns]
    X = data.drop(columns=cols_to_drop)
    y = data['HCLM']
    
    # 4. 划分数据集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"【3. 数据集划分】")
    print(f"训练集: {X_train.shape[0]} 行")
    print(f"测试集: {X_test.shape[0]} 行")
    print("-" * 50)

    # 5. 特征预处理管道
    numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

    numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='mean'))])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='passthrough'
    )

    # 6. 定义模型列表 (顺序决定了生成图片的后缀 a, b, c, d, e)
    models = {
        'LinearRegression': LinearRegression(),
        'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
    }
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    execution_results = []
    
    # 7. 模型训练与评估
    print("【4. 模型训练开始】")
    for (name, model), color_code in zip(models.items(), colors):
        print(f"正在运行模型: {name}...")
        
        model_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', model)
        ])
        
        model_pipeline.fit(X_train, y_train)
        y_pred = model_pipeline.predict(X_test)
        
        if 'Time' in X_test.columns:
            y_pred[X_test['Time'] == 0] = 0
        
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mape = np.mean(np.abs((y_test - y_pred) / (y_test.replace(0, 1e-9)))) * 100
        
        y_test_mean = y_test.mean()
        cv_rmse = (rmse / y_test_mean) * 100 if y_test_mean != 0 else np.inf
        
        metrics = {'R2': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'Cv(RMSE)': cv_rmse}
        
        execution_results.append({
            'name': name,
            'metrics': metrics,
            'y_pred': y_pred,
            'color': color_code
        })

    # 8. 计算全局Y轴范围 (统一对比图坐标)
    print("\n【5. 正在按照论文标准命名生成图表...】")
    all_values = list(y_test[:30].values)
    for result in execution_results:
        all_values.extend(result['y_pred'][:30])
    
    global_min = min(all_values)
    global_max = max(all_values)
    padding = (global_max - global_min) * 0.1
    y_axis_range = (global_min - padding, global_max + padding)

    # 9. 绘图与结果汇总 (按照论文编号格式命名)
    suffixes = ['a', 'b', 'c', 'd', 'e']  # 对应5个模型
    all_metrics_df = []
    
    for idx, result in enumerate(execution_results):
        suffix = suffixes[idx]
        
        # 按照要求构建文件名
        radar_filename = f"5-4{suffix}.png"
        scatter_filename = f"5-5{suffix}.png"
        case_filename = f"5-6{suffix}.png"

        # 生成并保存图片
        draw_metrics_radar(radar_filename, result['metrics'], result['color'])
        draw_regression_scatter(scatter_filename, y_test, result['y_pred'], result['color'])
        draw_case_comparison(case_filename, y_test, result['y_pred'], result['color'], y_range=y_axis_range)

        print(f"[{result['name']}] 生成 -> 雷达图:{radar_filename}, 评估图:{scatter_filename}, 案例图:{case_filename}")

        res_entry = result['metrics'].copy()
        res_entry['Model'] = result['name']
        all_metrics_df.append(res_entry)

    # 10. 保存结果CSV
    cols_order = ['R2', 'MAE', 'MSE', 'RMSE', 'MAPE', 'Cv(RMSE)']
    df_results = pd.DataFrame(all_metrics_df).set_index('Model')[cols_order]
    
    print("\n【6. 所有模型评估结果】")
    print(df_results.round(4))
    
    csv_save_path = os.path.join(output_dir, 'Model_Comparison_Results.csv')
    df_results.to_csv(csv_save_path)
    print(f"\n结果表格已保存至: {csv_save_path}")
    print(f"所有图片已按照 5-4a~e, 5-5a~e, 5-6a~e 规范保存至 [{output_dir}] 文件夹中。")
    print("运行顺利结束！")

except FileNotFoundError as e:
    print(f"错误: {e}")
except Exception as e:
    print(f"发生了一个错误: {e}")
    print(traceback.format_exc())

输出目录已存在: 机器
字体加载成功: SimSun & Times New Roman
--------------------------------------------------
【1. 原始数据统计】
原始数据形状 (行, 列): (46110, 22)
原始缺失值总数: 46110
--------------------------------------------------
【2. 数据清洗报告】
过滤物理范围不合理行数: 1060
删除目标值(HCLM)缺失行数: 0
清洗后数据形状 (行, 列): (45050, 22)
剩余有效样本数: 45050
--------------------------------------------------
【3. 数据集划分】
训练集: 36040 行
测试集: 9010 行
--------------------------------------------------
【4. 模型训练开始】
正在运行模型: LinearRegression...
正在运行模型: RandomForest...
正在运行模型: GradientBoosting...
正在运行模型: XGBoost...
正在运行模型: LightGBM...

【5. 正在按照论文标准命名生成图表...】
[LinearRegression] 生成 -> 雷达图:5-4a.png, 评估图:5-5a.png, 案例图:5-6a.png
[RandomForest] 生成 -> 雷达图:5-4b.png, 评估图:5-5b.png, 案例图:5-6b.png
[GradientBoosting] 生成 -> 雷达图:5-4c.png, 评估图:5-5c.png, 案例图:5-6c.png
[XGBoost] 生成 -> 雷达图:5-4d.png, 评估图:5-5d.png, 案例图:5-6d.png
[LightGBM] 生成 -> 雷达图:5-4e.png, 评估图:5-5e.png, 案例图:5-6e.png

【6. 所有模型评估结果】
                      R2     MAE       MSE     RMSE     MAPE  Cv(RMSE)
Model              

In [7]:
import os
import sys
import re  # 用于正则清洗括号
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import rcParams
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
import optuna
import shap
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

# === 彻底屏蔽 Jupyter 环境下的红色警告 ===
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', message='.IProgress not found.')
warnings.filterwarnings('ignore', module='tqdm')

# ------------------------------------------------------------------
# 0. 自动化归档 (基础目录)
# ------------------------------------------------------------------

BASE_OUTPUT_DIR = "SHAP_Optimization_Results_CNXX" 
SUMMARY_DIR = "最终汇总报表XX"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)
print(f"📁 主输出目录: ./{BASE_OUTPUT_DIR}/")
print(f"📊 汇总报表目录: ./{SUMMARY_DIR}/\n")

GLOBAL_SUMMARY_DATA = []

# ------------------------------------------------------------------
# 1. 字体精细设置 (中文宋体 + 英文新罗马)
# ------------------------------------------------------------------

# 尝试自动寻找系统中的宋体
try:
    # Windows 常见宋体路径
    font_cn_path = "C:/Windows/Fonts/simsun.ttc"
    if not os.path.exists(font_cn_path):
        # Mac/Linux 备选（如果没有simsun，可能需要手动指定）
        font_cn_path = "simsun.ttc" 
    
    # 定义中文字体 (用于坐标轴标题、图例中文)
    font_song = fm.FontProperties(fname=font_cn_path, size=14)
    font_song_small = fm.FontProperties(fname=font_cn_path, size=12)
    
    # 定义英文字体 (用于刻度、特征名)
    font_en = fm.FontProperties(family='Times New Roman', size=13)
    font_en_bold = fm.FontProperties(family='Times New Roman', size=14, weight='bold')

except:
    print("⚠️ 未找到宋体文件，将使用默认字体，可能导致中文乱码。")
    font_song = font_en = fm.FontProperties(size=12)

# 全局配置，优先衬线体
config = {
    "font.family": 'serif',
    "font.serif": ['Times New Roman', 'SimSun'],
    "mathtext.fontset": 'stix',
    "axes.unicode_minus": False
}
rcParams.update(config)

# ------------------------------------------------------------------
# 2. 数据精准清洗 (含列名去括号)
# ------------------------------------------------------------------

print("正在加载与预处理数据...")
try:
    data = pd.read_csv('JQXX.csv')
    
    # === 新增：正则清洗列名，去除括号及内容 ===
    # 例如: "Velocity (m/s)" -> "Velocity"
    clean_columns = []
    for col in data.columns:
        # 去除 () 或 （） 及其内部内容
        new_col = re.sub(r'[\(（].*?[\)）]', '', col).strip()
        clean_columns.append(new_col)
    data.columns = clean_columns
    print(f"✅ 列名清洗完成（已去除单位括号）。")

    def safe_convert(val):
        if pd.isna(val): return np.nan
        if isinstance(val, (int, float)): return float(val)
        val_str = str(val).translate(str.maketrans('', '', '[](){} ')).strip()
        try:
            return float(val_str)
        except ValueError:
            return np.nan 

    cols_to_convert = [c for c in data.columns if c not in ['Run_ID', 'Medium_Type']]
    for col in cols_to_convert:
        data[col] = data[col].apply(safe_convert)
    
    data.dropna(axis=1, thresh=len(data)*0.5, inplace=True)
    data.dropna(inplace=True)
    
    if len(data) == 0:
        print("\n❌ 严重错误: 数据被完全删空！")
        sys.exit()
        
    if 'HCLM' in data.columns:
        data = data[(data['HCLM'] >= 0) & (data['HCLM'] <= 120)]
    
    if 'Medium_Type' in data.columns:
        data = pd.get_dummies(data, columns=['Medium_Type'], prefix='Medium')

    drop_cols = ['Run_ID', 'HCLM']
    cols_to_drop = [c for c in drop_cols if c in data.columns]
    X = data.drop(columns=cols_to_drop).astype(float)
    y = data['HCLM'].astype(float)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_test_shap = X_test.sample(n=min(len(X_test), 500), random_state=42)

    print(f"✅ 数据准备就绪！特征维度: {X.shape[1]}\n")

except Exception as e:
    print(f"数据处理阶段出现意外错误：{e}")
    sys.exit()

# ------------------------------------------------------------------
# 3. 核心工具
# ------------------------------------------------------------------

def safe_compute_shap(model, model_name, X_sample, X_background):
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample, check_additivity=False)
        return shap_values if not isinstance(shap_values, list) else shap_values[0], X_sample
    except:
        def predict_wrapper(data_in):
            df_temp = pd.DataFrame(data_in, columns=X_sample.columns).astype(float)
            return model.predict(df_temp)
        bg_data = X_background[X_sample.columns].sample(min(len(X_background), 100))
        explainer = shap.KernelExplainer(predict_wrapper, shap.kmeans(bg_data, 10))
        shap_values = explainer.shap_values(X_sample, silent=True)
        return shap_values if not isinstance(shap_values, list) else shap_values[0], X_sample

def evaluate_performance(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if np.sum(mask) > 0 else np.nan
    mean_obs = np.mean(y_true)
    cv_rmse = (rmse / mean_obs) * 100 if mean_obs != 0 else np.nan
    return {"R2": r2, "RMSE": rmse, "MSE": mse, "MAE": mae, "MAPE (%)": mape, "Cv(RMSE) (%)": cv_rmse}

# ------------------------------------------------------------------
# 4. 绘图模块 (全面汉化版)
# ------------------------------------------------------------------

def draw_summary_shap(shap_values, X_shap_df, model_name, out_dir, n_trials):
    fig = plt.figure(figsize=(10, 6), dpi=300)
    # 绘制基础图
    shap.summary_plot(shap_values, X_shap_df, max_display=10, show=False)
    
    ax = plt.gca()
    # 1. 修改横轴标签为宋体中文
    ax.set_xlabel("SHAP值 (对模型输出的影响)", fontproperties=font_song)
    
    # 2. 修改纵轴字体为 New Roman
    plt.yticks(fontproperties=font_en)
    plt.xticks(fontproperties=font_en)
    
    # 3. 汉化 Colorbar (High/Low -> 高/低)
    # 获取当前 Figure 的所有 Axes，通常最后一个是 Colorbar
    fig_axes = plt.gcf().axes
    if len(fig_axes) > 1:
        cbar_ax = fig_axes[-1]
        # 覆盖原有的 High/Low 文本
        cbar_ax.set_ylabel("特征值", fontproperties=font_song, labelpad=10)
        # 尝试寻找并修改刻度文本，或者直接画文本覆盖
        # 简单粗暴法：在 Colorbar 轴的两端写字
        cbar_ax.text(1.1, 0.02, "低", transform=cbar_ax.transAxes, ha='left', va='center', fontproperties=font_song)
        cbar_ax.text(1.1, 0.98, "高", transform=cbar_ax.transAxes, ha='left', va='center', fontproperties=font_song)
        # 隐藏原有的 High/Low (如果能找到的话，通常它们是 text 对象)
        # 这里不做隐藏，直接覆盖通常效果也可以，或者因为 shap 库更新可能位置不同，留给自动布局

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{model_name}_{n_trials}_1_Summary.png'), dpi=300, bbox_inches='tight')
    plt.close()

def draw_bar_donut_shap_perfect(shap_values, X_shap_df, model_name, out_dir, n_trials, max_display=8):
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    sort_inds = np.argsort(mean_abs_shap)[::-1][:max_display]
    top_values = mean_abs_shap[sort_inds]
    top_labels = X_shap_df.columns[sort_inds]
    top_fractions = top_values / np.sum(top_values)
    colors_hex = ['#FCD863', '#FCE38A', '#FEF0BB', '#FDF8E1', '#E8E1E6', '#D6CADD', '#BAA8C8', '#A28BC1']
    current_colors = colors_hex[:len(top_values)]

    fig = plt.figure(figsize=(18, 7), dpi=300)
    gs = GridSpec(1, 3, width_ratios=[2.2, 2.0, 0.8], wspace=0.25)

    # --- 左侧：条形图 ---
    ax_bar = fig.add_subplot(gs[0])
    y_pos = np.arange(len(top_labels))
    bars = ax_bar.barh(y_pos, top_values[::-1], color=current_colors[::-1], height=0.7, edgecolor='none')
    ax_bar.set_xlim(0, np.max(top_values) * 1.15)
    for bar in bars:
        ax_bar.text(bar.get_width() + np.max(top_values)*0.02, bar.get_y() + bar.get_height()/2,
                f"{bar.get_width():.2f}", va='center', ha='left', fontproperties=font_en) # 数字用英文
    for y in y_pos: ax_bar.axhline(y, color='#e0e0e0', linestyle='--', linewidth=0.8, zorder=0)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(top_labels[::-1], fontproperties=font_en) # 特征名用英文
    # 汉化 X 轴标签
    ax_bar.set_xlabel("SHAP重要性 (平均绝对SHAP值)", fontproperties=font_song) 
    
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.spines['left'].set_linewidth(2)
    ax_bar.spines['bottom'].set_linewidth(2)

    # --- 中间：圆环图 ---
    ax_pie = fig.add_subplot(gs[1])
    wedges, texts = ax_pie.pie(top_fractions, colors=current_colors, startangle=90, wedgeprops=dict(width=0.45, edgecolor='white', linewidth=1.5))
    ax_pie.set_aspect('equal')
    # ... (原有圆环图逻辑不变，只修改字体) ...
    labels_data = []
    for i, p in enumerate(wedges):
        ang = (p.theta2 - p.theta1)/2. + p.theta1
        labels_data.append({'x': np.cos(np.deg2rad(ang)), 'y': np.sin(np.deg2rad(ang)), 'ang': ang, 'frac': top_fractions[i]})
    left_side = sorted([d for d in labels_data if d['x'] <= 0], key=lambda val: val['y'])
    right_side = sorted([d for d in labels_data if d['x'] > 0], key=lambda val: val['y'])
    def adjust_y(side_data, min_dist=0.18): 
        if len(side_data) <= 1: return
        for i in range(1, len(side_data)):
            if side_data[i]['y'] - side_data[i-1]['y'] < min_dist:
                side_data[i]['y'] = side_data[i-1]['y'] + min_dist
        if side_data[-1]['y'] > 1.1:
            shift = side_data[-1]['y'] - 1.1
            for d in side_data: d['y'] -= shift
    adjust_y(left_side)
    adjust_y(right_side)
    for d in left_side + right_side:
        orig_x, orig_y = np.cos(np.deg2rad(d['ang'])), np.sin(np.deg2rad(d['ang']))
        sign_x = -1 if orig_x <= 0 else 1
        adj_x = 1.25 * sign_x 
        ax_pie.annotate(
            f"{d['frac']*100:.1f}%", xy=(orig_x, orig_y), xytext=(adj_x, d['y']),
            horizontalalignment="right" if sign_x < 0 else "left", verticalalignment="center", fontproperties=font_en, 
            arrowprops=dict(arrowstyle="-", color="#888888", linewidth=1.0, connectionstyle=f"angle,angleA={0 if sign_x < 0 else 180},angleB={d['ang']}")
        )

    # --- 右侧：图例 (汉化) ---
    ax_leg = fig.add_subplot(gs[2])
    ax_leg.axis('off') 
    x_offset = 0.15 
    # 汉化标题
    ax_leg.text(x_offset, 0.92, "高贡献", fontproperties=font_song, ha='left')
    for i in range(len(top_labels)):
        y_cood = 0.80 - i * 0.11
        ax_leg.add_patch(Rectangle(xy=(x_offset, y_cood), width=0.18, height=0.07, color=current_colors[i]))
        ax_leg.text(x_offset + 0.28, y_cood + 0.035, top_labels[i], va='center', fontproperties=font_en_bold) # 特征名加粗英文
    ax_leg.text(x_offset, 0.80 - len(top_labels)*0.11, "低贡献", fontproperties=font_song, ha='left')

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{model_name}_{n_trials}_2_Bar_Donut.png'), dpi=300, bbox_inches='tight')
    plt.close()

def draw_heatmap_shap(shap_values, X_shap_df, model_name, out_dir, n_trials):
    try:
        exp = shap.Explanation(values=shap_values, data=X_shap_df.values, feature_names=list(X_shap_df.columns))
        fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
        shap.plots.heatmap(exp, max_display=10, show=False)
        # 汉化
        ax.set_xlabel("样本 (按函数值排序)", fontproperties=font_song)
        ax.set_ylabel("模型输出值", fontproperties=font_song)
        # 设置刻度为英文
        plt.xticks(fontproperties=font_en)
        plt.yticks(fontproperties=font_en)
        
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'{model_name}_{n_trials}_3_Heatmap.png'), dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"      [警告] {model_name} 热力图生成跳过: {e}")
        plt.close()

def draw_dependence_shap(shap_values, X_shap_df, model_name, out_dir, n_trials):
    try:
        top_feature = X_shap_df.columns[np.argsort(np.abs(shap_values).mean(0))[-1]]
        fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
        
        # === 修改点：关闭交互作用 (interaction_index=None) ===
        shap.dependence_plot(top_feature, shap_values, X_shap_df, ax=ax, show=False, interaction_index=None)
        
        # 汉化坐标轴
        ax.set_xlabel(top_feature, fontproperties=font_en) # X轴是特征名，用英文
        ax.set_ylabel("SHAP值", fontproperties=font_song) # Y轴标题用中文
        
        # 刻度设置
        plt.xticks(fontproperties=font_en)
        plt.yticks(fontproperties=font_en)
        
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'{model_name}_{n_trials}_4_Dependence.png'), dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"      [警告] {model_name} 依赖图生成跳过: {e}")
        plt.close()

# ------------------------------------------------------------------
# 5. 模型寻优逻辑
# ------------------------------------------------------------------

def process_model(model_type, X_tr, y_tr, X_te, y_te, X_shap, n_trials, out_dir):
    def objective(trial):
        if model_type == 'RandomForest':
            param = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 5, 20),
            }
            model = RandomForestRegressor(**param, n_jobs=-1, random_state=42)
        elif model_type == 'XGBoost':
            param = {
                'n_estimators': trial.suggest_int('n_estimators', 200, 800),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
            }
            model = xgb.XGBRegressor(objective='reg:squarederror', **param, n_jobs=-1, random_state=42)
        elif model_type == 'LightGBM':
            param = {
                'n_estimators': trial.suggest_int('n_estimators', 200, 800),
                'num_leaves': trial.suggest_int('num_leaves', 20, 100),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
            }
            model = lgb.LGBMRegressor(**param, n_jobs=-1, random_state=42, verbose=-1)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        if 'Time' in X_te.columns: preds[X_te['Time'] == 0] = 0
        return np.sqrt(mean_squared_error(y_te, preds))

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials) 

    best_params = study.best_params
    train_params = best_params.copy()
    train_params['random_state'] = 42
    train_params['n_jobs'] = -1
    if model_type == 'LightGBM': train_params['verbose'] = -1

    if model_type == 'RandomForest': final_model = RandomForestRegressor(**train_params)
    elif model_type == 'XGBoost': final_model = xgb.XGBRegressor(objective='reg:squarederror', **train_params)
    elif model_type == 'LightGBM': final_model = lgb.LGBMRegressor(**train_params)

    final_model.fit(X_tr, y_tr)
    final_preds = final_model.predict(X_te)
    if 'Time' in X_te.columns: final_preds[X_te['Time'] == 0] = 0
    
    metrics = evaluate_performance(y_te, final_preds)
    
    print(f"👉 [{model_type}] 最优 RMSE: {metrics['RMSE']:.4f}")
    
    shap_vals, X_shap_df = safe_compute_shap(final_model, model_type, X_shap, X_tr)
    if shap_vals is not None:
        draw_summary_shap(shap_vals, X_shap_df, model_type, out_dir, n_trials)
        draw_bar_donut_shap_perfect(shap_vals, X_shap_df, model_type, out_dir, n_trials)
        draw_heatmap_shap(shap_vals, X_shap_df, model_type, out_dir, n_trials)
        draw_dependence_shap(shap_vals, X_shap_df, model_type, out_dir, n_trials)

    return {'Model': model_type, 'Trials': n_trials, **metrics, **best_params}

# ------------------------------------------------------------------
# 6. 批量寻优执行
# ------------------------------------------------------------------

trial_counts = [10, 20, 50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]

for n_trials in trial_counts:
    out_dir = os.path.join(BASE_OUTPUT_DIR, f'Trials_{n_trials}')
    os.makedirs(out_dir, exist_ok=True)
    sep = '*' * 45
    print(f"{sep}\n🚀 启动任务 ---> 寻优: {n_trials} 次\n{sep}")
    
    current_results = {}
    for model_name in ['RandomForest', 'XGBoost', 'LightGBM']:
        res_dict = process_model(model_name, X_train, y_train, X_test, y_test, X_test_shap, n_trials, out_dir)
        current_results[model_name] = res_dict
        GLOBAL_SUMMARY_DATA.append(res_dict)

    with open(os.path.join(out_dir, f'Report_{n_trials}.txt'), 'w', encoding='utf-8') as f:
        f.write(f"============ 全指标评估报告 (寻优:{n_trials}次) ============\n")
        sorted_models = sorted(current_results.values(), key=lambda x: x['RMSE'])
        for res in sorted_models:
            f.write(f"【{res['Model']}】\n")
            f.write(f"  [性能指标]\n")
            for k in ['R2', 'RMSE', 'MAE', 'MSE', 'MAPE (%)', 'Cv(RMSE) (%)']:
                f.write(f"    {k:<12}: {res[k]:.4f}\n")
            f.write(f"  [最优参数]\n")
            ignore = ['Model', 'Trials', 'R2', 'RMSE', 'MAE', 'MSE', 'MAPE (%)', 'Cv(RMSE) (%)']
            for k, v in res.items():
                if k not in ignore: f.write(f"    * {k}: {v}\n")
            f.write("\n")

# ------------------------------------------------------------------
# 7. 生成最终汇总 Excel
# ------------------------------------------------------------------

print(f"\n{'-'*60}\n💾 正在生成最终汇总 Excel...\n{'-'*60}")

if GLOBAL_SUMMARY_DATA:
    df_summary = pd.DataFrame(GLOBAL_SUMMARY_DATA)
    first_cols = ['Model', 'Trials', 'R2', 'RMSE', 'MAE', 'MSE', 'MAPE (%)', 'Cv(RMSE) (%)']
    param_cols = [c for c in df_summary.columns if c not in first_cols]
    df_summary = df_summary[first_cols + param_cols]
    excel_path = os.path.join(SUMMARY_DIR, 'Optimization_Summary.xlsx')
    try:
        df_summary.to_excel(excel_path, index=False, float_format="%.4f")
        print(f"🎉 全部任务完成！")
        print(f"1. 各组详细汉化图表: ./{BASE_OUTPUT_DIR}/")
        print(f"2. 最终数据表: {excel_path}")
    except Exception as e:
        print(f"⚠️ 保存Excel失败: {e} (请 pip install openpyxl)")
else:
    print("⚠️ 无数据生成。")

📁 主输出目录: ./SHAP_Optimization_Results_CNXX/
📊 汇总报表目录: ./最终汇总报表XX/

正在加载与预处理数据...
✅ 列名清洗完成（已去除单位括号）。
✅ 数据准备就绪！特征维度: 22

*********************************************
🚀 启动任务 ---> 寻优: 10 次
*********************************************
👉 [RandomForest] 最优 RMSE: 0.7724
👉 [XGBoost] 最优 RMSE: 0.6862
👉 [LightGBM] 最优 RMSE: 0.7116
*********************************************
🚀 启动任务 ---> 寻优: 20 次
*********************************************
👉 [RandomForest] 最优 RMSE: 0.7853
👉 [XGBoost] 最优 RMSE: 0.6947
👉 [LightGBM] 最优 RMSE: 0.7143
*********************************************
🚀 启动任务 ---> 寻优: 50 次
*********************************************
👉 [RandomForest] 最优 RMSE: 0.7658
👉 [XGBoost] 最优 RMSE: 0.6612
👉 [LightGBM] 最优 RMSE: 0.6727
*********************************************
🚀 启动任务 ---> 寻优: 100 次
*********************************************
👉 [RandomForest] 最优 RMSE: 0.7652
👉 [XGBoost] 最优 RMSE: 0.6374
👉 [LightGBM] 最优 RMSE: 0.7024
*********************************************
🚀 启动任务 ---> 寻优: 200 次
*